1. Importaciones y configuración inicial

In [1]:
import requests
import pandas as pd
import os

os.makedirs("data/raw", exist_ok=True)

2. Descagrar datos Eurostat con api

In [2]:
#Lista de países con código y nombre completo

#Dataset ligero para extraer países:
url_paises = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/demo_pjan"

response = requests.get(url_paises)
data = response.json()

#Extraer códigos y nombres:
dimensiones = data["dimension"]

geo_codes = list(dimensiones["geo"]["category"]["index"].keys())
geo_labels = dimensiones["geo"]["category"]["label"]

#Reconstruir tabla:
records = []

for g in geo_codes:
    records.append({"country_code": g,
                    "country_name": geo_labels[g]})

#Crear df:
countries = pd.DataFrame(records)

#Filtrar solo países zona Euro:
filtro_countries = ["AT","BE","CY","DE","EE","ES","FI","FR","EL","IE","IT","LT","LU","LV","MT","NL","PT","SI","SK","HR"]
countries = countries[countries["country_code"].isin(filtro_countries)].reset_index(drop=True)

#Guardar CSV:
output_path = os.path.join("data", "raw", "countries.csv")
countries.to_csv(output_path, index=False)

countries

,country_code,country_name
0,BE,Belgium
1,DE,Germany
2,EE,Estonia
3,IE,Ireland
4,EL,Greece
5,ES,Spain
6,FR,France
7,HR,Croatia
8,IT,Italy
9,CY,Cyprus


In [3]:
#Gini coefficient of equivalised disposable income

url_gini = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tessi190"

response = requests.get(url_gini)
data = response.json()

#Extraer dimensiones:
dimensiones = data["dimension"]

geo = list(dimensiones["geo"]["category"]["index"].keys())
time = list(dimensiones["time"]["category"]["index"].keys())
age = list(dimensiones["age"]["category"]["index"].keys())
statinfo = list(dimensiones["statinfo"]["category"]["index"].keys())

values = data["value"]

#Reconstruir tabla completa:
records = []
idx = 0

for g in geo:
    for t in time:
        for a in age:
            for s in statinfo:
                value = values.get(str(idx), None)
                records.append({"country": g,
                                "year": int(t),
                                "age": a,
                                "statinfo": s,
                                "gini": value})
                idx += 1

#Crear df:
gini = pd.DataFrame(records)

#Filtrar información que realmento necesito:
gini = gini[(gini["age"] == "TOTAL") & (gini["statinfo"] == "GINI_HND")]

filtro_countries = ["AT","BE","CY","DE","EE","ES","FI","FR","EL","IE","IT","LT","LU","LV","MT","NL","PT","SI","SK","HR"]
gini = gini[gini["country"].isin(filtro_countries)]


#Quedarme con columnas necesarias:
gini = gini[["country", "year", "gini"]].reset_index(drop=True)

#Guardar CSV:
output_path = os.path.join("data", "raw", "gini.csv")
gini.to_csv(output_path, index=False)

gini.head()

,country,year,gini
0,BE,2014,25.9
1,BE,2015,26.2
2,BE,2016,26.3
3,BE,2017,26.1
4,BE,2018,25.7


In [4]:
#Home ownership rate

url_housing = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/ilc_lvho02"

response = requests.get(url_housing)
data = response.json()

#Extraer dimensiones:
dimensiones = data["dimension"]

freq   = list(dimensiones["freq"]["category"]["index"].keys())
incgrp = list(dimensiones["incgrp"]["category"]["index"].keys())
hhtyp  = list(dimensiones["hhtyp"]["category"]["index"].keys())
tenure = list(dimensiones["tenure"]["category"]["index"].keys())
unit   = list(dimensiones["unit"]["category"]["index"].keys())
geo    = list(dimensiones["geo"]["category"]["index"].keys())
time   = list(dimensiones["time"]["category"]["index"].keys())

values = data["value"]

#Reconstruir tabla completa:
records = []
idx = 0

for f in freq:
    for inc in incgrp:
        for hh in hhtyp:
            for ten in tenure:
                for u in unit:
                    for g in geo:
                        for t in time:
                            value = values.get(str(idx), None)
                            records.append({
                                "freq": f,
                                "incgrp": inc,
                                "hhtyp": hh,
                                "tenure": ten,
                                "unit": u,
                                "country": g,
                                "year": int(t),
                                "home_ownership_rate": value
                            })
                            idx += 1

#Crear df:
housing = pd.DataFrame(records)

#Filtrar información que realmento necesito:
housing = housing[(housing["freq"] == "A") & (housing["unit"] == "PC") & (housing["incgrp"] == "TOTAL") & (housing["hhtyp"] == "TOTAL") & (housing["tenure"] == "OWN")]

filtro_countries = ["AT","BE","CY","DE","EE","ES","FI","FR","EL","IE","IT","LT","LU","LV","MT","NL","PT","SI","SK","HR"]
housing = housing[housing["country"].isin(filtro_countries)]

#Quedarme con columnas necesarias:
housing = housing[["country", "year", "home_ownership_rate"]].reset_index(drop=True)

#Guardar CSV:
output_path = os.path.join("data", "raw", "housing.csv")
housing.to_csv(output_path, index=False)

housing.head()

,country,year,home_ownership_rate
0,BE,2003,72.7
1,BE,2004,72.2
2,BE,2005,72.2
3,BE,2006,73.7
4,BE,2007,72.9


In [5]:
#Unemployment rate

url_unemployment = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tps00203"

response = requests.get(url_unemployment)
data = response.json()

#Extraer dimensiones:
dimensiones = data["dimension"]

geo = list(dimensiones["geo"]["category"]["index"].keys())
time = list(dimensiones["time"]["category"]["index"].keys())
unit = list(dimensiones["unit"]["category"]["index"].keys())
freq = list(dimensiones["freq"]["category"]["index"].keys())
age  = list(dimensiones["age"]["category"]["index"].keys())
sex  = list(dimensiones["sex"]["category"]["index"].keys())

values = data["value"]

#Reconstruir tabla completa:
records = []
idx = 0

for f in freq:
    for a in age:
        for u in unit:
            for s in sex:
                for g in geo:
                    for t in time:
                        value = values.get(str(idx), None)
                        records.append({
                            "country": g,
                            "year": int(t),
                            "unit": u,
                            "unemployment_rate": value
                        })
                        idx += 1

#Crear df:
unemployment = pd.DataFrame(records)

#Filtrar información que realmento necesito:
unemployment = unemployment[unemployment["unit"] == "PC_ACT"]

filtro_countries = ["AT","BE","CY","DE","EE","ES","FI","FR","EL","IE","IT","LT","LU","LV","MT","NL","PT","SI","SK","HR"]
unemployment = unemployment[unemployment["country"].isin(filtro_countries)]

#Quedarme con columnas necesarias:
unemployment = unemployment[["country", "year", "unemployment_rate"]].reset_index(drop=True)

#Guardar CSV:
output_path = os.path.join("data", "raw", "unemployment.csv")
unemployment.to_csv(output_path, index=False)

unemployment.head()

,country,year,unemployment_rate
0,BE,2014,8.7
1,BE,2015,8.7
2,BE,2016,7.9
3,BE,2017,7.2
4,BE,2018,6.0


In [6]:
#Early leavers from education and training

url_education = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/sdg_04_10"

response = requests.get(url_education)
data = response.json()

#Extraer dimensiones:
dimensiones = data["dimension"]

freq = list(dimensiones["freq"]["category"]["index"].keys())
unit = list(dimensiones["unit"]["category"]["index"].keys())
wstatus = list(dimensiones["wstatus"]["category"]["index"].keys())
age = list(dimensiones["age"]["category"]["index"].keys())
sex = list(dimensiones["sex"]["category"]["index"].keys())
geo = list(dimensiones["geo"]["category"]["index"].keys())
time = list(dimensiones["time"]["category"]["index"].keys())

values = data["value"]

#Reconstruir tabla completa:
records = []
idx = 0

for f in freq:
    for u in unit:
        for ws in wstatus:
            for a in age:
                for s in sex:
                    for g in geo:
                        for t in time:
                            value = values.get(str(idx), None)
                            records.append({"country": g,
                                            "year": int(t),
                                            "sex": s,
                                            "early_leavers": value})
                            idx += 1

#Crear df:
education = pd.DataFrame(records)

#Filtrar información que realmento necesito:
education = education[(education["sex"] == "T")]

filtro_countries = ["AT","BE","CY","DE","EE","ES","FI","FR","EL","IE","IT","LT","LU","LV","MT","NL","PT","SI","SK","HR"]
education = education[education["country"].isin(filtro_countries)]


#Quedarme con columnas necesarias:
education = education[["country", "year", "early_leavers"]].reset_index(drop=True)

#Guardar CSV:
output_path = os.path.join("data", "raw", "education.csv")
education.to_csv(output_path, index=False)

education.head()

,country,year,early_leavers
0,BE,2000,13.8
1,BE,2001,13.8
2,BE,2002,14.1
3,BE,2003,14.3
4,BE,2004,13.1


In [7]:
#Healthy life years at birth

url_health = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tps00150"

response = requests.get(url_health)
data = response.json()

#Extraer dimensiones:
dimensiones = data["dimension"]

freq = list(dimensiones["freq"]["category"]["index"].keys())
unit = list(dimensiones["unit"]["category"]["index"].keys())
sex = list(dimensiones["sex"]["category"]["index"].keys())
indic_he = list(dimensiones["indic_he"]["category"]["index"].keys())
geo = list(dimensiones["geo"]["category"]["index"].keys())
time = list(dimensiones["time"]["category"]["index"].keys())

values = data["value"]

#Reconstruir tabla completa:
records = []
idx = 0

for f in freq:
    for u in unit:
        for s in sex:
            for ind in indic_he:
                for g in geo:
                    for t in time:
                        value = values.get(str(idx), None)
                        records.append({"country": g,
                                        "year": int(t),
                                        "sex": s,
                                        "healthy_life_years": value})
                        idx += 1

#Crear df:
health = pd.DataFrame(records)

#Filtrar información que realmento necesito:
health = health[(health["sex"] == "T")]

filtro_countries = ["AT","BE","CY","DE","EE","ES","FI","FR","EL","IE","IT","LT","LU","LV","MT","NL","PT","SI","SK","HR"]
health = health[health["country"].isin(filtro_countries)]

#Quedarme con columnas necesarias:
health = health[["country", "year", "healthy_life_years"]].reset_index(drop=True)

#Guardar CSV:
output_path = os.path.join("data", "raw", "health.csv")
health.to_csv(output_path, index=False)

health.head()

,country,year,healthy_life_years
0,BE,2012,64.6
1,BE,2013,63.9
2,BE,2014,64.1
3,BE,2015,64.2
4,BE,2016,63.7


In [8]:
#GDP per capita (real terms)

url_gdpcapita = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/sdg_08_10"

response = requests.get(url_gdpcapita)
data = response.json()

#Extraer dimensiones:
dimensiones = data["dimension"]

geo = list(dimensiones["geo"]["category"]["index"].keys())
time = list(dimensiones["time"]["category"]["index"].keys())

values = data["value"]

#Reconstruir tabla completa:
records = []
idx = 0

for g in geo:
    for t in time:
        value = values.get(str(idx), None)
        records.append({"country": g,
                        "year": int(t),
                        "gdp_capita_real": value})
        idx += 1

#Crear df:
gdp_capita = pd.DataFrame(records)

#Filtrar información que realmento necesito:
filtro_countries = ["AT","BE","CY","DE","EE","ES","FI","FR","EL","IE","IT","LT","LU","LV","MT","NL","PT","SI","SK","HR"]
gdp_capita = gdp_capita[gdp_capita["country"].isin(filtro_countries)]


#Ordenar columnas:
gdp_capita = gdp_capita[["country", "year", "gdp_capita_real"]].reset_index(drop=True)

#Guardar CSV:
output_path = os.path.join("data", "raw", "gdp_capita.csv")
gdp_capita.to_csv(output_path, index=False)

gdp_capita.head()

,country,year,gdp_capita_real
0,BE,2000,35370.0
1,BE,2001,35640.0
2,BE,2002,36080.0
3,BE,2003,36300.0
4,BE,2004,37440.0


In [9]:
#Infaltion rate

url_inflation = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tec00118"

response = requests.get(url_inflation)
data = response.json()

#Extraer dimensiones:
dimensiones = data["dimension"]

geo = list(dimensiones["geo"]["category"]["index"].keys())
time = list(dimensiones["time"]["category"]["index"].keys())

values = data["value"]

#Reconstruir tabla completa:
records = []
idx = 0

for g in geo:
    for t in time:
        value = values.get(str(idx), None)
        records.append({"country": g,
                        "year": int(t),
                        "inflation_rate": value})
        idx += 1

#Crear df:
inflation = pd.DataFrame(records)

#Filtrar información que realmento necesito:
filtro_countries = ["AT","BE","CY","DE","EE","ES","FI","FR","EL","IE","IT","LT","LU","LV","MT","NL","PT","SI","SK","HR"]
inflation = inflation[inflation["country"].isin(filtro_countries)]

#Guardar CSV:
output_path = os.path.join("data", "raw", "inflation.csv")
inflation.to_csv(output_path, index=False)

inflation.head()

,country,year,inflation_rate
48,BE,2014,0.5
49,BE,2015,0.6
50,BE,2016,1.8
51,BE,2017,2.2
52,BE,2018,2.3
